# Tracking clustered regions

Link cluster detections using box-centre distance and the Hungarian assignment algorithm. This exploratory tracker is separate from tracker.py, which uses an annotated target and local search.


## 1. Setup

Load a gesture recording and shared clustering helpers from foveanet.py. Notebook 01 section 9 compares the clustering methods.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.optimize import linear_sum_assignment

import foveanet as fn

dataset = fn.load_gesture()
events, label = dataset[0]
x, y, t, p = fn.unpack(events)

print("recordings:", len(dataset))
print("events in this recording:", len(events))
print("duration (ms):", (t.max() - t.min()) / 1000)
print("clustering method:", fn.METHOD, "| k =", fn.K, "| window =", fn.WINDOW_US / 1000, "ms")

## 2. Candidate detections

Keep three clusters per frame, ranked by event count, rather than selecting a single region immediately.


In [ ]:
frames = list(fn.iter_frames(x, y, t))
mid = frames[len(frames) // 2]
s_mid, xs, ys, ts = mid

dets = fn.detections(xs, ys, ts)
img = fn.frame_image(xs, ys)
colours = ["cyan", "yellow", "magenta"]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.imshow(img, cmap="hot")
for d, colour in zip(dets, colours):
    bx0, by0, bx1, by1 = d["box"]
    ax.add_patch(
        patches.Rectangle(
            (bx0, by0), bx1 - bx0, by1 - by0, lw=2, edgecolor=colour, facecolor="none"
        )
    )
    ax.text(bx0, by0 - 2, f"{d['size']} ev", color=colour, fontsize=8)
ax.set_title(f"top {len(dets)} detections, {fn.METHOD} k={fn.K}")
ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Tracker

Match detections to tracks by centre distance. Reject matches beyond max_dist; smooth matched boxes with alpha; retain unmatched tracks briefly and initialise unmatched detections as new tracks.


In [ ]:
def make_tracker(max_dist=30.0, min_hits=3, max_misses=5, alpha=0.2):
    return {
        "tracks": [],
        "next_id": 0,
        "history": {},
        "fovea_id": None,
        "cfg": dict(max_dist=max_dist, min_hits=min_hits, max_misses=max_misses, alpha=alpha),
    }


def track_step(state, dets):
    cfg, tracks = state["cfg"], state["tracks"]
    boxes = [d["box"] for d in dets]

    matched = {}
    if tracks and boxes:
        cost = np.zeros((len(tracks), len(boxes)))
        for i, tr in enumerate(tracks):
            cx, cy = fn.box_centre(tr["box"])
            for j, b in enumerate(boxes):
                bx, by = fn.box_centre(b)
                cost[i, j] = np.hypot(cx - bx, cy - by)
        rows, cols = linear_sum_assignment(cost)
        matched = {i: j for i, j in zip(rows, cols) if cost[i, j] <= cfg["max_dist"]}

    used = set(matched.values())
    a = cfg["alpha"]
    for i, tr in enumerate(tracks):
        if i in matched:
            b = boxes[matched[i]]
            tr["box"] = tuple(a * np.array(b) + (1 - a) * np.array(tr["box"]))
            tr["raw_box"] = b
            tr["size"] = dets[matched[i]]["size"]
            tr["hits"] += 1
            tr["misses"] = 0
        else:
            tr["misses"] += 1
        tr["age"] += 1

    for j, b in enumerate(boxes):
        if j not in used:
            tracks.append(
                {
                    "id": state["next_id"],
                    "box": tuple(map(float, b)),
                    "raw_box": b,
                    "size": dets[j]["size"],
                    "hits": 1,
                    "misses": 0,
                    "age": 1,
                }
            )
            state["next_id"] += 1

    state["tracks"] = [tr for tr in tracks if tr["misses"] <= cfg["max_misses"]]
    for tr in state["tracks"]:
        state["history"].setdefault(tr["id"], []).append(fn.box_centre(tr["box"]))
    return [tr for tr in state["tracks"] if tr["hits"] >= cfg["min_hits"]]


def fovea_track(state, confirmed, margin=1.5):
    if not confirmed:
        state["fovea_id"] = None
        return None
    by_id = {tr["id"]: tr for tr in confirmed}
    current = state.get("fovea_id")
    if current in by_id:
        best = max(confirmed, key=lambda tr: tr["hits"])
        if best["id"] != current and best["hits"] > by_id[current]["hits"] * margin:
            state["fovea_id"] = best["id"]
    else:
        state["fovea_id"] = max(confirmed, key=lambda tr: (tr["hits"], tr["size"]))["id"]
    return by_id[state["fovea_id"]]

## 4. Process a recording

Inspect selected boxes and track IDs at six times. A stable ID is a continuity measure, not proof that the same physical target was followed.


In [ ]:
tracker = make_tracker()
timeline = []

for s, xs_, ys_, ts_ in fn.iter_frames(x, y, t):
    dets_ = fn.detections(xs_, ys_, ts_)
    if not dets_:
        continue
    confirmed = track_step(tracker, dets_)
    fov = fovea_track(tracker, confirmed)
    timeline.append(
        {
            "t": s,
            "xs": xs_,
            "ys": ys_,
            "raw": dets_[0]["box"],
            "fovea": None if fov is None else tuple(fov["box"]),
            "id": None if fov is None else fov["id"],
            "hits": None if fov is None else fov["hits"],
            "n_confirmed": len(confirmed),
        }
    )

covered = [f for f in timeline if f["fovea"] is not None]
print(
    f"frames: {len(timeline)}, with a confirmed fovea: {len(covered)} "
    f"({100 * len(covered) / len(timeline):.0f}%)"
)
print(f"tracks created: {tracker['next_id']}")

idx = np.linspace(0, len(covered) - 1, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, i in zip(axes.ravel(), idx):
    f = covered[i]
    ax.imshow(fn.frame_image(f["xs"], f["ys"]), cmap="hot")
    bx0, by0, bx1, by1 = [int(round(v)) for v in f["fovea"]]
    ax.add_patch(
        patches.Rectangle(
            (bx0, by0), bx1 - bx0, by1 - by0, lw=2, edgecolor="cyan", facecolor="none"
        )
    )
    ax.set_title(
        f"t = {(f['t'] - t.min()) / 1000:.0f} ms, track {f['id']}, {f['hits']} hits", fontsize=9
    )
    ax.axis("off")
plt.suptitle("Tracked fovea")
plt.tight_layout()
plt.show()

## 5. Centre trajectories

Compare raw selected-box centres with smoothed tracked centres. Examine jumps and lag as well as average displacement.


In [ ]:
raw_c = np.array([fn.box_centre(f["raw"]) for f in timeline])
trk_c = np.array([fn.box_centre(f["fovea"]) if f["fovea"] else (np.nan, np.nan) for f in timeline])
ms = (np.array([f["t"] for f in timeline]) - t.min()) / 1000

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, dim, name in zip(axes, [0, 1], ["x", "y"]):
    ax.plot(ms, raw_c[:, dim], color="0.6", lw=1, label="raw per-frame salient box")
    ax.plot(ms, trk_c[:, dim], color="tab:blue", lw=1.8, label="tracked fovea")
    ax.set_ylabel(f"{name} centre (px)")
axes[0].legend(fontsize=8)
axes[1].set_xlabel("time (ms)")
plt.suptitle("Fovea centre, before and after tracking")
plt.tight_layout()
plt.show()

## 6. Selection stability

Sweep hysteresis and smoothing to compare identity switches, displacement and event capture. A low average displacement alone can hide occasional large jumps.


In [ ]:
def jump_stats(centres):
    c = np.array([p for p in centres if p is not None])
    if len(c) < 3:
        return None
    d = np.hypot(*(c[1:] - c[:-1]).T)
    return d.mean(), np.percentile(d, 95), (d > 30).mean()


def capture_of(box, xs_, ys_):
    b = [int(round(v)) for v in box]
    return ((xs_ >= b[0]) & (xs_ <= b[2]) & (ys_ >= b[1]) & (ys_ <= b[3])).mean()


rng = np.random.default_rng(0)
targets = np.array(dataset.targets)
all_idx = np.arange(len(dataset))
sample = []
for cls in sorted(set(targets)):
    sample += rng.choice(all_idx[targets == cls], 2, replace=False).tolist()

cache = []
for i in sample:
    ev_, _ = dataset[i]
    xi, yi, ti, _ = fn.unpack(ev_)
    seq = []
    for s, xs_, ys_, ts_ in fn.iter_frames(xi, yi, ti):
        d = fn.detections(xs_, ys_, ts_)
        if d:
            seq.append((d, xs_, ys_))
    cache.append(seq)
print(f"recordings: {len(cache)}, frames: {sum(len(s) for s in cache)}")

raw_j = [jump_stats([fn.box_centre(d[0]["box"]) for d, _, _ in seq]) for seq in cache]
raw_c = [np.mean([capture_of(d[0]["box"], xs_, ys_) for d, xs_, ys_ in seq]) for seq in cache]

print(f"\n{'selection':>24}{'mean':>8}{'p95':>8}{'>30px':>8}{'capture':>9}{'id switch':>11}")
print(
    f"{'raw per-frame':>24}{np.mean([r[0] for r in raw_j]):>8.1f}"
    f"{np.mean([r[1] for r in raw_j]):>8.1f}{100 * np.mean([r[2] for r in raw_j]):>7.1f}%"
    f"{np.mean(raw_c):>9.2f}{'-':>11}"
)

for name, margin in [("most hits", None), ("sticky incumbent", 1.5)]:
    stats, caps, switches = [], [], []
    for seq in cache:
        tr = make_tracker()
        centres, cap_seq, ids = [], [], []
        for d, xs_, ys_ in seq:
            confirmed_ = track_step(tr, d)
            if margin is None:
                pool = [z for z in confirmed_ if z["misses"] == 0]
                f = max(pool, key=lambda z: (z["hits"], z["size"])) if pool else None
            else:
                f = fovea_track(tr, confirmed_, margin=margin)
            if f:
                centres.append(fn.box_centre(f["box"]))
                cap_seq.append(capture_of(f["box"], xs_, ys_))
                ids.append(f["id"])
            else:
                centres.append(None)
        st = jump_stats(centres)
        if st and ids:
            stats.append(st)
            caps.append(np.mean(cap_seq))
            switches.append(sum(1 for a, b in zip(ids, ids[1:]) if a != b) / len(ids))
    print(
        f"{name:>24}{np.mean([s[0] for s in stats]):>8.1f}"
        f"{np.mean([s[1] for s in stats]):>8.1f}{100 * np.mean([s[2] for s in stats]):>7.1f}%"
        f"{np.mean(caps):>9.2f}{100 * np.mean(switches):>10.1f}%"
    )

## 7. Track lifetime

Plot confirmed-track lifetimes and compare their durations. Long-lived tracks indicate persistence but do not establish target identity without annotations.


In [ ]:
lifetimes = sorted((tr["hits"] for tr in tracker["tracks"]), reverse=True)
all_hist = [len(v) for v in tracker["history"].values()]

fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4))
axl.hist(all_hist, bins=20, color="tab:blue", alpha=0.8)
axl.set_xlabel("track lifetime (frames)")
axl.set_ylabel("number of tracks")
axl.set_title("track lifetimes")

longest = max(tracker["history"].items(), key=lambda kv: len(kv[1]))
path = np.array(longest[1])
axr.plot(path[:, 0], path[:, 1], lw=1.5, color="tab:blue")
axr.scatter(path[0, 0], path[0, 1], c="green", s=40, label="start")
axr.scatter(path[-1, 0], path[-1, 1], c="red", s=40, label="end")
axr.set_xlim(0, 128)
axr.set_ylim(128, 0)
axr.set_aspect("equal")
axr.legend(fontsize=8)
axr.set_title(f"path of longest track (id {longest[0]}, {len(path)} frames)")
plt.tight_layout()
plt.show()

print("longest few track lifetimes (frames):", lifetimes[:5])

## 8. Related experiments

Notebook 03 evaluates persistence within the GMM. Notebook 06 measures tracking against annotated ground truth; classify.py evaluates the resulting crop representations.
